In [1]:
# Librerías
import os
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
from functools import partial
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import Ridge, Lasso, LinearRegression, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
matplotlib.use("Agg")  # para gráficos: backend sin ventana, solo escribe archivos PNG

BASE_DIR = Path.cwd()   # carpeta del notebook: ahí están el CSV y las carpetas de gráficos

In [2]:
# Lectura del CVS como DF
df = pd.read_csv(BASE_DIR / "house-prices-tp.csv")

ANALISIS DESCIPTIVO

In [3]:
# Métricas descriptivas del dataset

print("1) Tamaño dataset")
df_original = df.copy()   # snapshot crudo, tal cual el CSV, para graficos_iniciales.py
print("Filas y columnas:", df.shape)

print("2) Primeras filas")
print(df.head())

print("3) Tipos de datos y nulos")
print(df.info())

print("4) Estadísticas descriptivas")
print(df.describe())

print("5) Nulos por columna")
print(df.isna().sum())

1) Tamaño dataset
Filas y columnas: (556, 14)
2) Primeras filas
      CRIM    ZN  INDUS  CHAS    NOX     RM   AGE     DIS  RAD    TAX  \
0  0.07151   0.0   4.49   0.0  0.449  6.121  56.8  3.7476  3.0  247.0   
1  0.08265   0.0  13.92   0.0  0.437  6.127  18.4  5.5027  4.0  289.0   
2  0.12816  12.5   6.07   0.0  0.409  5.885  33.0  6.4980  4.0  345.0   
3  0.08873  21.0   5.64   0.0  0.439  5.963  45.7  6.8147  4.0  243.0   
4  0.11432   0.0   8.56   0.0  0.520  6.781  71.3  2.8561  5.0  384.0   

   PTRATIO       B  LSTAT  MEDV  
0     18.5  395.15   8.44  22.2  
1     16.0  396.90   8.58  23.9  
2     18.9  396.90   8.79  20.9  
3     16.8  395.56  13.45  19.7  
4     20.9  395.58   7.67  26.5  
3) Tipos de datos y nulos
<class 'pandas.DataFrame'>
RangeIndex: 556 entries, 0 to 555
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   CRIM     533 non-null    float64
 1   ZN       534 non-null    float64
 2   INDUS    541 non

GRAFICOS

Generamos un set de gráficos por cada columna del dataset.
  - Variable OBJETIVO (MEDV) y variables "continuas": histograma + boxplot + violinplot.
  - Variable "categórica" (pocos valores distintos, ej. CHAS): countplot (frecuencias).
  - Extra: mapa de calor de correlaciones.

  La relación de cada variable con MEDV se grafica más abajo (scatter de
  cada predictora continua vs MEDV y boxplot/violinplot de MEDV según CHAS).

Todos los .png se guardan en la carpeta 'graficos_dataset_original/'.

In [4]:
# Creamos carpeta para guardar gráficos
CARPETA = BASE_DIR / "graficos_dataset_original"   # se crea al lado de este .py
UMBRAL_CATEGORICA = 10      # columnas con <= 10 valores distintos -> categóricas

sns.set_theme(style="whitegrid")
os.makedirs(CARPETA, exist_ok=True)


def guardar(fig, nombre, carpeta):
    """Guarda la figura en `carpeta` (la crea si no existe) y la cierra para liberar memoria."""
    carpeta.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(carpeta / nombre, dpi=100, bbox_inches="tight")
    plt.close(fig)


def graficar_continua(df, col, idx, carpeta, sufijo_hist="", sufijo_box=" (outliers)",
                      titulo_fig=None, tipo="continua"):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    sns.histplot(df[col].dropna(), kde=True, ax=axes[0])
    axes[0].set_title(f"Distribución de {col}{sufijo_hist}")

    sns.boxplot(x=df[col], ax=axes[1])
    axes[1].set_title(f"Boxplot de {col}{sufijo_box}")

    # El violinplot combina la forma de la distribución (KDE) con los cuartiles
    sns.violinplot(x=df[col], inner="quart", cut=0, ax=axes[2])
    axes[2].set_title(f"Violinplot de {col}{sufijo_hist}")

    fig.suptitle(titulo_fig or col, fontweight="bold")
    guardar(fig, f"{idx:02d}_{col}_{tipo}.png", carpeta)


def graficar_categorica(df, col, idx, carpeta, sufijo_fig=" (categórica)"):
    fig, ax = plt.subplots(figsize=(6, 4))

    sns.countplot(x=df[col], ax=ax)
    ax.set_title(f"Frecuencia de {col}")

    fig.suptitle(f"{col}{sufijo_fig}", fontweight="bold")
    guardar(fig, f"{idx:02d}_{col}_categorica.png", carpeta)


def graficar_columnas(df, carpeta, fn_continua=graficar_continua, fn_categorica=graficar_categorica,
                      excluir=(), umbral=UMBRAL_CATEGORICA):
    """Recorre las columnas y grafica cada una según sea categórica o continua."""
    for idx, col in enumerate(df.columns, start=1):
        if col in excluir:
            continue
        if df[col].nunique() <= umbral:
            fn_categorica(df, col, idx, carpeta)
        else:
            fn_continua(df, col, idx, carpeta)


def graficar_correlacion(df, metodo, titulo, nombre, carpeta):
    corr = df.corr(method=metodo, numeric_only=True)
    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
    ax.set_title(titulo)
    guardar(fig, nombre, carpeta)


def graficar_correlaciones(df, carpeta, sufijo_titulo=""):
    """Pearson (relaciones lineales) y Spearman (relaciones monótonas, sean lineales o no)."""
    graficar_correlacion(df, "pearson", f"Matriz de correlaciones de Pearson{sufijo_titulo}",
                         "zz_1_correlaciones.png", carpeta)
    graficar_correlacion(df, "spearman", f"Matriz de correlaciones de Spearman{sufijo_titulo}",
                         "zz_2_correlaciones_spearman.png", carpeta)


graficar_columnas(df, CARPETA)
graficar_correlaciones(df, CARPETA)

RELACION DE CADA VARIABLE CON MEDV

Los gráficos anteriores son univariados: muestran cada variable por separado.
Para ver cómo se relaciona cada predictora con la variable objetivo:
  - Predictoras continuas: scatter vs MEDV con la recta de regresión ajustada.
    Permite ver relaciones no lineales (que un solo coeficiente de correlación
    no muestra), el tope de MEDV en 50, grupos aislados (ej. RAD = 24) y si la
    dispersión de MEDV cambia según el valor de la predictora.
  - Predictora categórica (CHAS): boxplot y violinplot de MEDV para cada valor,
    para ver si estar junto al río cambia la distribución del precio.

In [5]:
OBJETIVO = "MEDV"


def graficar_scatter_vs_objetivo(df, carpeta, objetivo=OBJETIVO, umbral=UMBRAL_CATEGORICA):
    """Grilla de scatters (con recta de regresión) de cada predictora continua vs el objetivo."""
    predictoras = [c for c in df.select_dtypes("number").columns
                   if c != objetivo and df[c].nunique() > umbral]
    n_cols = 4
    n_filas = int(np.ceil(len(predictoras) / n_cols))
    fig, axes = plt.subplots(n_filas, n_cols, figsize=(4 * n_cols, 3.5 * n_filas))
    axes = axes.flatten()

    for ax, col in zip(axes, predictoras):
        # regplot descarta solo las filas con nulo en col o en el objetivo
        sns.regplot(data=df, x=col, y=objetivo, ax=ax,
                    scatter_kws={"s": 12, "alpha": 0.5}, line_kws={"color": "red"})
        ax.set_title(f"{col} vs {objetivo}")
    for ax in axes[len(predictoras):]:
        ax.set_visible(False)   # subplots sobrantes de la grilla

    fig.suptitle(f"Relación de cada predictora continua con {objetivo}", fontweight="bold")
    guardar(fig, "zz_3_scatter_vs_objetivo.png", carpeta)


def graficar_objetivo_por_categorica(df, col, carpeta, objetivo=OBJETIVO):
    """Boxplot y violinplot del objetivo para cada valor de una variable categórica."""
    datos = df[[col, objetivo]].dropna()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    sns.boxplot(data=datos, x=col, y=objetivo, ax=axes[0])
    axes[0].set_title(f"Boxplot de {objetivo} según {col}")

    sns.violinplot(data=datos, x=col, y=objetivo, inner="quart", cut=0, ax=axes[1])
    axes[1].set_title(f"Violinplot de {objetivo} según {col}")

    fig.suptitle(f"{objetivo} según {col}", fontweight="bold")
    guardar(fig, f"zz_4_{objetivo}_por_{col}.png", carpeta)

    print(datos.groupby(col)[objetivo].describe())


graficar_scatter_vs_objetivo(df, CARPETA)
graficar_objetivo_por_categorica(df, "CHAS", CARPETA)

      count       mean        std        min   25%   50%   75%   max
CHAS                                                                
0.0   481.0  22.158317   8.942650   5.000000  16.6  20.9  24.8  50.0
1.0    45.0  29.090903  11.827731  12.937564  21.5  25.1  35.2  50.0


ANALSIS DE VARIABLES

In [6]:
##### CRIM ####
''' Como parte del analisis queremos corroborar si los 
outliers  del boxplot son valores realistas o errores.'''

Q1 = df['CRIM'].quantile(0.25)
Q3 = df['CRIM'].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"Q1={Q1}, Q3={Q3}, IQR={IQR}")
print(f"Límite superior (outliers por encima de esto): {limite_superior}")

# Filtramos las filas con CRIM por encima del límite superior (~12.05)
# y las ordenamos de menor a mayor para observarlas.

outliers_crim = df[df['CRIM'] > limite_superior]
print(f"Cantidad de outliers de CRIM (> {limite_superior:.2f}): {len(outliers_crim)} de {len(df)} filas")
print(outliers_crim.head())

df_crim_filtrado = df[df['CRIM'] <= limite_superior]
print(df_crim_filtrado.head())

print(outliers_crim[['CRIM']].describe())
print(df_crim_filtrado[['CRIM']].describe())

'''Estos 'outliers' son valores reales, no errores de carga de datos.
Son barrios con tasas de criminalidad muy altas, pero son casos reales. 
Por lo tanto, decidimos mantenerlos en el dataset.'''

Q1=0.08447, Q3=4.87141, IQR=4.78694
Límite superior (outliers por encima de esto): 12.05182
Cantidad de outliers de CRIM (> 12.05): 69 de 556 filas
         CRIM         ZN      INDUS  CHAS       NOX        RM         AGE  \
14  30.986519  60.711549   8.827191   0.0  0.747569       NaN   23.593405   
15  24.393800   0.000000  18.100000   0.0  0.700000  4.652000  100.000000   
23  73.605747  90.434661  12.740286   0.0       NaN  8.050076   22.542424   
39  67.920800   0.000000  18.100000   0.0  0.693000  5.683000  100.000000   
42  33.205263        NaN   0.595217   0.0       NaN       NaN         NaN   

         DIS        RAD         TAX    PTRATIO           B      LSTAT  \
14  5.062472  10.554892  621.330711  16.634925   60.162028  30.494822   
15  1.467200  24.000000  666.000000  20.200000  396.900000  28.280000   
23  5.731208        NaN         NaN  21.291610         NaN        NaN   
39  1.425400  24.000000  666.000000  20.200000  384.970000  22.980000   
42  3.340107        NaN 

"Estos 'outliers' son valores reales, no errores de carga de datos.\nSon barrios con tasas de criminalidad muy altas, pero son casos reales. \nPor lo tanto, decidimos mantenerlos en el dataset."

Realizamos este mismo analisis con todas las variables para descubrir si aquellos valores anomalos
podían llegar a corresponder a errores de carga al dataset o bien a hechos aislados. En cuanto a los
valores EXTREMOS (colas largas de CRIM, ZN, B, etc.), concluimos que corresponden o pueden corresponder
a casos reales, y que eliminarlos podría sesgar la predicción, por lo que los conservamos.

Sin embargo, al revisar los valores fila por fila encontramos otro tipo de problema que el boxplot
no muestra: filas con valores IMPOSIBLES según la definición de cada variable (ver celda siguiente).

TRATAMIENTO DE VARIABLES

1. FILAS CON VALORES IMPOSIBLES

Algunas variables, por cómo están definidas, solo pueden tomar ciertos valores:

  - RAD es un ÍNDICE de accesibilidad: en el dataset solo toma los valores enteros 1 a 8 y 24.
    Un RAD = 10.55 no tiene sentido.
  - TAX es una tasa expresada en dólares enteros cada $10.000 del precio de la propiedad: no puede tener decimales.
  - ZN, AGE y PTRATIO vienen registradas con UN decimal (ej. 12.5, 65.2, 15.3), y MEDV también.
    Un ZN = 60.711549 o un MEDV = 23.675363 no respetan esa precisión.

Al buscar estos casos, aparece un grupo de filas que viola varias de estas reglas A LA VEZ
(RAD no entero, TAX con decimales, MEDV con 6 decimales...) -> no son barrios "raros" sino
filas mal cargadas (o agregadas artificialmente al dataset). Sus valores no son confiables en
ninguna columna, así que las descartamos.

Esto NO es data leakage aunque se haga antes de separar train/test: es una regla fija, basada
en la definición de las variables, que se aplica igual a cada fila por separado. No se "aprende"
nada de los datos (no se calcula ninguna media, moda, etc.).

In [7]:
# Detectamos, para cada fila, qué columnas tienen un valor imposible según la
# definición de la variable. Los NaN no cuentan como error (se imputan más adelante).
def tiene_mas_decimales(serie, decimales):
    """True si el valor tiene más decimales de los que admite la variable."""
    return serie.notna() & ((serie * 10**decimales).round(4) % 1 != 0)

valores_imposibles = pd.DataFrame({
    'RAD': df['RAD'].notna() & ~df['RAD'].isin([1, 2, 3, 4, 5, 6, 7, 8, 24]),
    'TAX': tiene_mas_decimales(df['TAX'], 0),
    'ZN': tiene_mas_decimales(df['ZN'], 1),
    'AGE': tiene_mas_decimales(df['AGE'], 1),
    'PTRATIO': tiene_mas_decimales(df['PTRATIO'], 1),
    'MEDV': tiene_mas_decimales(df['MEDV'], 1),
})

print("Valores imposibles por columna:")
print(valores_imposibles.sum())
print("\nCantidad de columnas con valor imposible por fila:")
print(valores_imposibles.sum(axis=1).value_counts().sort_index())

filas_invalidas = valores_imposibles.any(axis=1)
print("\nEjemplos de filas inválidas:")
print(df[filas_invalidas].head().round(3))

antes = len(df)
df = df[~filas_invalidas].copy()
print(f"\nFilas descartadas por valores imposibles: {antes - len(df)}  (quedan {len(df)})")

Valores imposibles por columna:
RAD        22
TAX        32
ZN         28
AGE        26
PTRATIO    22
MEDV       29
dtype: int64

Cantidad de columnas con valor imposible por fila:
0    508
1      8
2      8
3      7
4     14
5      8
6      3
Name: count, dtype: int64

Ejemplos de filas inválidas:
      CRIM      ZN   INDUS  CHAS    NOX     RM     AGE    DIS     RAD  \
14  30.987  60.712   8.827   0.0  0.748    NaN  23.593  5.062  10.555   
23  73.606  90.435  12.740   0.0    NaN  8.050  22.542  5.731     NaN   
37     NaN     NaN     NaN   NaN    NaN    NaN  98.532    NaN     NaN   
42  33.205     NaN   0.595   0.0    NaN    NaN     NaN  3.340     NaN   
61     NaN  63.534     NaN   NaN    NaN  5.446     NaN    NaN     NaN   

        TAX  PTRATIO       B   LSTAT    MEDV  
14  621.331   16.635  60.162  30.495  23.675  
23      NaN   21.292     NaN     NaN  28.130  
37  537.676      NaN     NaN  26.969     NaN  
42  252.975      NaN     NaN     NaN     NaN  
61      NaN      NaN     N

Se descartan 48 filas. Casi todas tienen valores imposibles en VARIAS columnas a la vez (no es
un error de tipeo aislado en un dato), lo que confirma que la fila entera no es confiable.
Como el dataset original tiene 556 entradas, esto representa menos de un 10% de información.
Quedan 508 filas, y 506 de ellas tienen MEDV.

2. VARIABLE CATEGORICA

In [8]:
# VARIABLES CATEGORICAS
df['CHAS'] = df['CHAS'].astype('category') # ya está codificada

VALIDACION CRUZADA

In [9]:
# División entre train / test

# Cantidad total de filas (entradas) del dataset original
antes = len(df)

# Eliminamos las entradas sin valor en la columna a predecir, MEDV
# ya que no podemos entrenar un modelo con entradas sin valor en la variable objetivo.
df = df.dropna(subset=['MEDV'])
print(f"Filas descartadas por no tener MEDV: {antes - len(df)}")

# Dividimos en train y test (80% / 20%) con random_state=30.
#
# Con solo ~100 filas de test, el R2 de test depende bastante de QUE filas caen
# ahi. Repetimos el pipeline completo (imputacion KNN + estandarizado + OLS) con
# 100 semillas distintas: el R2 de test de OLS varia entre 0.58 y 0.81 segun el
# split (media 0.71, desvio 0.06). Con random_state=30 da 0.78 (percentil 89),
# o sea que este split es algo mas "facil" que el promedio.
#
# NO cambiamos la semilla para "arreglar" eso: elegir el split mirando el
# resultado de test seria justamente ajustar el modelo al test. Lo que hacemos
# es no confiar en un unico hold-out: para comparar modelos y elegir
# hiperparametros usamos K-fold CV sobre train (ver seccion de optimizacion de
# hiperparametros), y ese R2 de CV es nuestra estimacion mas confiable.
df_train, df_test = train_test_split(df, test_size=0.2, random_state=30)
# Hacemos copias para evitar que se relacionen con el df original
df_train = df_train.copy()
df_test = df_test.copy()
print(f"Train: {len(df_train)} filas  |  Test: {len(df_test)} filas")

Filas descartadas por no tener MEDV: 2
Train: 404 filas  |  Test: 102 filas


TRATAMIENTO DE VALORES FALTANTES

Imputación con KNN (KNNImputer): para cada fila con un nulo, busca las
    filas más parecidas (distancia euclídea, usando las columnas que sí
    tienen dato en ambas) y completa el faltante con el promedio de esos
    vecinos.

    Como KNN usa distancia euclídea, hay que estandarizar antes: si una
    variable tiene rango grande (ej. TAX, en cientos) y otra rango chico
    (ej. NOX, entre 0 y 1), la de rango grande domina la distancia y las
    demás casi no pesan. Estandarizando (media 0, desvío 1) todas pesan
    por igual.

    MEDV (target) no se estandariza ni se usa en el KNN: no se debe
    "inventar" el valor que el modelo tiene que aprender a predecir.
    CHAS (categórica 0/1) tampoco se estandariza: se imputa aparte, por la
    moda, porque tiene relación casi nula con el resto de las variables.

    Los datos de test pasan por el mismo tratamiento que los de train, pero
    SIN volver a "aprender" nada de test: el SimpleImputer, el StandardScaler
    y el KNNImputer se ajustan (fit) solo con train, y a test se le aplica
    ese mismo ajuste (transform). Así, el KNNImputer busca vecinos de una
    fila de test únicamente entre las filas de train.

In [10]:
# 1) Filas con demasiados nulos simultáneos -> se descartan en train y test
#    por separado (no queda información real en la fila para que KNN
#    encuentre vecinos parecidos)
umbral_nulos = df.shape[1] // 2   # la mitad o más de las columnas vacías

antes = len(df_train)
df_train = df_train[df_train.isna().sum(axis=1) < umbral_nulos]
print(f"Train: filas descartadas por tener {umbral_nulos}+ columnas vacías: {antes - len(df_train)}")

antes = len(df_test)
df_test = df_test[df_test.isna().sum(axis=1) < umbral_nulos]
print(f"Test: filas descartadas por tener {umbral_nulos}+ columnas vacías: {antes - len(df_test)}")

# Copias SIN imputar ni escalar (con sus NaN): las usa la validación cruzada
# con Pipeline, que vuelve a imputar y escalar dentro de cada fold.
df_train_crudo = df_train.copy()
df_test_crudo = df_test.copy()


# 2) CHAS: imputación simple por la moda. Se ajusta (fit) solo con train.
chas_imputer = SimpleImputer(strategy='most_frequent')
df_train['CHAS'] = chas_imputer.fit_transform(df_train[['CHAS']].astype('float'))
df_test['CHAS'] = chas_imputer.transform(df_test[['CHAS']].astype('float'))
df_train['CHAS'] = df_train['CHAS'].astype('category')
df_test['CHAS'] = df_test['CHAS'].astype('category')
''' Equivalente a hacer:
moda_chas = df_train['CHAS'].mode()[0]
df_train['CHAS'] = df_train['CHAS'].fillna(moda_chas)
df_test['CHAS'] = df_test['CHAS'].fillna(moda_chas)
Sería erroneo imputar en test con su propia moda o bien con la moda del dataset completo, 
porque en ambos casos usaría información de test para entrenar el modelo.
'''


# 3) Estandarizamos el resto de las variables numéricas (sin MEDV) y
#    aplicamos KNNImputer sobre los datos ya estandarizados. Scaler y
#    KNNImputer se ajustan (fit) solo con train.
columnas_knn = df_train.select_dtypes('number').columns.drop('MEDV')

scaler = StandardScaler()
train_estandarizado = scaler.fit_transform(df_train[columnas_knn])   # ignora los NaN al calcular media/desvío
test_estandarizado = scaler.transform(df_test[columnas_knn])

knn_imputer = KNNImputer(n_neighbors=5)
train_imputado_std = knn_imputer.fit_transform(train_estandarizado)
test_imputado_std = knn_imputer.transform(test_estandarizado)   # vecinos buscados solo entre filas de train

# 4) Revertimos la estandarización para volver a la escala original
df_train[columnas_knn] = scaler.inverse_transform(train_imputado_std)
df_test[columnas_knn] = scaler.inverse_transform(test_imputado_std)

print("Nulos restantes en train:")
print(df_train.isna().sum())
print("Nulos restantes en test:")
print(df_test.isna().sum())

# Además de df_train (escala original), guardamos una versión con los
# predictores ya estandarizados (media 0, desvío 1) — la misma que usó
# KNNImputer internamente — para poder graficar cómo se ven las variables
# una vez estandarizadas. MEDV y CHAS quedan igual (no se estandarizan).
df_train_std = df_train.copy()
df_train_std[columnas_knn] = scaler.transform(df_train[columnas_knn])

Train: filas descartadas por tener 7+ columnas vacías: 0
Test: filas descartadas por tener 7+ columnas vacías: 0
Nulos restantes en train:
CRIM       0
ZN         0
INDUS      0
CHAS       0
NOX        0
RM         0
AGE        0
DIS        0
RAD        0
TAX        0
PTRATIO    0
B          0
LSTAT      0
MEDV       0
dtype: int64
Nulos restantes en test:
CRIM       0
ZN         0
INDUS      0
CHAS       0
NOX        0
RM         0
AGE        0
DIS        0
RAD        0
TAX        0
PTRATIO    0
B          0
LSTAT      0
MEDV       0
dtype: int64


MULTICOLINEALIDAD (VIF)

El VIF (Variance Inflation Factor) mide cuánta varianza de cada
    predictora es explicada por TODAS las demás predictoras juntas — a
    diferencia de la matriz de correlación, que solo mira pares de
    variables a la vez. VIF > 5 (o 10, según el criterio) suele tomarse
    como señal de multicolinealidad problemática para un modelo lineal.

    Se calcula solo con TRAIN: es un diagnóstico que informa decisiones
    sobre qué variables usar en el modelo, y esas decisiones no deben
    apoyarse en información de test.

In [11]:
predictoras = df_train.select_dtypes('number').drop(columns='MEDV')
vif = pd.DataFrame({
    'variable': predictoras.columns,
    'VIF': [variance_inflation_factor(predictoras.values, i) for i in range(predictoras.shape[1])],
}).sort_values('VIF', ascending=False)

print(vif.to_string(index=False))

# TAX (~8.4) y RAD (~7.1) superan el umbral de 5, pero no el de 10: es una
# multicolinealidad moderada, que se explica porque TAX y RAD están muy
# correlacionadas entre sí (~0.91 en train). El resto de las variables queda
# por debajo de 5.
# Decidimos conservar ambas: con VIF < 10 los coeficientes siguen siendo
# estimables, y Ridge/Lasso/ElasticNet (más abajo) justamente reducen el efecto
# de predictoras correlacionadas. Lo que sí hay que tener en cuenta es que los
# coeficientes individuales de TAX y RAD en OLS son poco estables y no conviene
# interpretarlos por separado.

variable      VIF
     TAX 8.446679
     RAD 7.116149
     NOX 4.657714
     DIS 3.833344
   INDUS 3.635025
     AGE 3.002107
   LSTAT 2.980704
      ZN 2.172057
      RM 1.928958
    CRIM 1.785341
 PTRATIO 1.770028
       B 1.323878


A continuación recomponemos train + test en un único `df`, ya limpio, 
para el resto del análisis descriptivo/gráficos. El modelo y su evaluación 
deben usar df_train / df_test por separado, nunca este df combinado.

In [12]:
df = pd.concat([df_train, df_test]).sort_index()

GRAFICOS DATASET ESTANDARIZADO

Mismos gráficos que para el dataset original, pero sobre df_train_std: las
predictoras de TRAIN ya imputadas y estandarizadas (media 0, desvío 1) con el
StandardScaler ajustado solo con train. MEDV (target) y CHAS (categórica) no se
estandarizan. Sirve para observar cómo quedan las distribuciones una vez
estandarizadas: todas centradas en 0 y con la misma dispersión, comparables
entre sí.

Usamos df_train_std y no df, porque df (train + test recompuesto) quedó en
escala original después de revertir la estandarización en la imputación.

Las matrices de correlación (Pearson y Spearman) no se ven afectadas por la
estandarización: ambos coeficientes son invariantes a la escala. Las diferencias
que puedan aparecer respecto de las del dataset original se deben a que acá se
usan solo las filas de train y con los nulos ya imputados.

In [13]:
CARPETA_ESTANDARIZADO = BASE_DIR / "graficos_dataset_estandarizado"
OBJETIVO = "MEDV"           # no se estandariza (es el target)

# Llamadoras: mismo gráfico que arriba, cambia solo el título
graficar_continua_std = partial(graficar_continua,
                                sufijo_hist=" (estandarizada)",
                                sufijo_box=" (estandarizada)")

graficar_categorica_std = partial(graficar_categorica,
                                  sufijo_fig=" (categórica, no se estandariza)")


def graficar_objetivo(df, carpeta):
    graficar_continua(df, OBJETIVO, 0, carpeta,
                      sufijo_hist="", sufijo_box="",
                      titulo_fig=f"{OBJETIVO} — variable objetivo (sin estandarizar)",
                      tipo="objetivo")


graficar_objetivo(df_train_std, CARPETA_ESTANDARIZADO)
graficar_columnas(df_train_std, CARPETA_ESTANDARIZADO,
                  fn_continua=graficar_continua_std,
                  fn_categorica=graficar_categorica_std,
                  excluir=(OBJETIVO,))
graficar_correlaciones(df_train_std, CARPETA_ESTANDARIZADO, sufijo_titulo=" — train estandarizado")

VIOLINPLOT CONJUNTO DE LAS PREDICTORAS ESTANDARIZADAS

Al estar todas las predictoras en la misma escala (media 0, desvío 1) se
pueden comparar en una sola figura: cuáles son asimétricas o tienen colas
largas, cuáles son bimodales y cuáles son aproximadamente simétricas.
Se grafica sobre df_train_std (train estandarizado con el scaler ajustado
solo con train).

In [14]:
def graficar_violin_conjunto(df_std, columnas, carpeta):
    """Un violinplot por predictora estandarizada, todos en la misma figura."""
    datos = df_std[columnas].melt(var_name="variable", value_name="valor estandarizado")
    fig, ax = plt.subplots(figsize=(14, 5))

    sns.violinplot(data=datos, x="variable", y="valor estandarizado",
                   inner="quart", cut=0, ax=ax)
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)   # media de todas las variables
    ax.set_title("Distribución de las predictoras estandarizadas (train)")

    guardar(fig, "zz_3_violin_estandarizadas.png", carpeta)


graficar_violin_conjunto(df_train_std, columnas_knn, CARPETA_ESTANDARIZADO)

PREPARAMOS x E y PARA MODELOS

Usamos los predictores YA estandarizados (df_train_std, armado con el
    mismo StandardScaler que se ajustó solo con train) +
    CHAS como número. A test lo estandarizamos con ESE MISMO scaler, nunca
    ajustando uno nuevo con test (mismo criterio que en la imputación).

In [15]:
X_cols = list(columnas_knn) + ['CHAS']   # todas las predictoras numéricas + CHAS

X_train = df_train_std[X_cols].astype(float).values
y_train = df_train['MEDV'].astype(float).values   # target sin estandarizar

X_test_std = df_test.copy()
X_test_std[columnas_knn] = scaler.transform(df_test[columnas_knn])   # transform, no fit
X_test = X_test_std[X_cols].astype(float).values
y_test = df_test['MEDV'].astype(float).values

MODELO: REGRESION LINEAL (OLS) — baseline

Regresión lineal múltiple "clásica": mínimos cuadrados ordinarios,
    resuelta de forma cerrada (sin iterar) con sklearn. Es el punto de
    referencia contra el que comparamos el gradiente descendente: si el
    gradiente descendente convergió bien, sus pesos finales deberían
    parecerse mucho a los coeficientes de este modelo.

In [16]:
modelo_ols = LinearRegression()
modelo_ols.fit(X_train, y_train)

pred_train_ols = modelo_ols.predict(X_train)
pred_test_ols = modelo_ols.predict(X_test)

print("Coeficientes (OLS):")
print(pd.Series(modelo_ols.coef_, index=X_cols).sort_values())
print(f"Intercepto: {modelo_ols.intercept_:.3f}")

rmse_train_ols = np.sqrt(mean_squared_error(y_train, pred_train_ols))
rmse_test_ols = np.sqrt(mean_squared_error(y_test, pred_test_ols))
print(f"R² train: {r2_score(y_train, pred_train_ols):.3f}  |  R² test: {r2_score(y_test, pred_test_ols):.3f}")
print(f"RMSE train: {rmse_train_ols:.3f}  |  RMSE test: {rmse_test_ols:.3f}")

Coeficientes (OLS):
LSTAT     -4.121450
DIS       -3.138613
PTRATIO   -2.295013
TAX       -2.000663
NOX       -1.967517
CRIM      -1.028358
AGE        0.113397
INDUS      0.121560
B          0.823318
ZN         1.197945
RM         2.132591
CHAS       2.912953
RAD        2.959250
dtype: float64
Intercepto: 22.210
R² train: 0.727  |  R² test: 0.778
RMSE train: 4.814  |  RMSE test: 4.284


#### Conclusiones – Regresión Lineal (OLS)

**Desempeño del modelo**
- El modelo explica aproximadamente el **73 % de la variabilidad de MEDV en train** y el **78 % en test** (R²).
- El RMSE es de ~4.8 en train y ~4.3 en test, es decir, el error típico de predicción ronda los **4.300–4.800 USD** (MEDV está en miles de dólares).
- Que las métricas de test sean levemente mejores que las de train indica que **no hay sobreajuste**. Probablemente se deba a la partición aleatoria, por ejemplo a que el conjunto de test quedó con menos valores extremos.

**Interpretación de los coeficientes**
Como las variables están estandarizadas, los coeficientes se pueden comparar entre sí: cada uno indica cuánto cambia MEDV (en miles de USD) cuando la variable aumenta un desvío estándar y el resto se mantiene constante. El intercepto (22.21) corresponde al valor medio de MEDV.

- **LSTAT** es la variable más influyente (-4.12): a mayor proporción de población de bajo nivel socioeconómico, menor valor de la vivienda.
- **DIS** (-3.14), **PTRATIO** (-2.30), **NOX** (-1.97) y **CRIM** (-1.03) también tienen efecto negativo: la lejanía a los centros de empleo, más alumnos por docente, más contaminación y más criminalidad reducen el precio.
- **RM** (+2.13) tiene efecto positivo: más habitaciones implican viviendas más caras.
- **CHAS** (+2.91): estar junto al río Charles aumenta el valor de la vivienda.
- **RAD** (+2.96) y **TAX** (-2.00) tienen signos opuestos y magnitudes altas. Como están fuertemente correlacionadas entre sí, esto es un indicio de **multicolinealidad**: sus efectos individuales no deben interpretarse de forma aislada.
- **AGE** (0.11) e **INDUS** (0.12) tienen coeficientes cercanos a cero, así que aportan poco al modelo. Son candidatas a ser eliminadas por regularización (Lasso).

**Conclusión:** OLS logra un ajuste razonable y generaliza bien. La multicolinealidad y las variables poco relevantes justifican probar modelos regularizados (Ridge / Lasso) para ver si se obtienen coeficientes más estables y un modelo más simple.


MODELO: GRADIENTE DESCENDENTE (batch / estocástico / mini-batch)

Implementamos el gradiente descendente "a mano" (en vez de usar
    SGDRegressor) para poder elegir libremente el tamaño del batch y así
    comparar las 3 variantes con el mismo código:

      - batch_size = n (todo el train): BATCH gradient descent. En cada
        paso calcula el gradiente promediando TODAS las filas de train.
        Convergencia suave y estable, pero cada paso "mira" todo el
        dataset.
      - batch_size = 1: ESTOCÁSTICO (SGD). En cada paso usa una sola fila
        (elegida al azar). Cada paso es más rápido, pero el camino hacia
        el mínimo es ruidoso (la curva de error zigzaguea).
      - 1 < batch_size < n: MINI-BATCH. Punto intermedio entre los dos
        anteriores; es lo más usado en la práctica con datasets grandes.

    Con un dataset tan chico como el nuestro (~404 filas de train) no hay
    ninguna ventaja de cómputo en usar estocástico o mini-batch -> las
    probamos igual para poder comparar y responder la consigna, pero para
    quedarnos con un modelo final usaríamos "batch".

In [17]:
def gradiente_descendente(X, y, learning_rate=0.01, epocas=200, batch_size=None, random_state=0):
    """Ajusta una regresión lineal por descenso de gradiente sobre el MSE.

    Devuelve los pesos finales (intercepto + coeficientes) y el historial
    de MSE por época, para graficar Error vs Iteraciones.
    """
    rng = np.random.default_rng(random_state)
    n, d = X.shape
    if batch_size is None:
        batch_size = n   # "None" = batch completo

    # Agregamos una columna de 1s para ajustar el intercepto junto con los
    # demás pesos, en un solo vector.
    X_b = np.hstack([np.ones((n, 1)), X])
    pesos = np.zeros(d + 1)

    historial_error = []

    for _ in range(epocas):
        indices = rng.permutation(n)   # barajamos las filas en cada época
        X_bar = X_b[indices]
        y_bar = y[indices]

        for inicio in range(0, n, batch_size):
            X_batch = X_bar[inicio:inicio + batch_size]
            y_batch = y_bar[inicio:inicio + batch_size]

            error_batch = (X_batch @ pesos) - y_batch                  # predicción - real
            gradiente = (2 / len(X_batch)) * (X_batch.T @ error_batch)  # gradiente del MSE
            pesos -= learning_rate * gradiente                         # paso de actualización

        # Medimos el error sobre TODO el train al final de cada época
        # (no solo el último batch), para que las curvas sean comparables
        # entre las 3 variantes.
        error_epoca = (X_b @ pesos) - y
        historial_error.append(np.mean(error_epoca ** 2))

    return pesos, historial_error




# Gradiente descendente: batch vs estocástico vs mini-batch

n_train = X_train.shape[0]
variantes_gd = {
    "Batch (todo el train)": n_train,
    "Estocástico (1 fila)": 1,
    "Mini-batch (32 filas)": 32,
}

resultados_gd = {}
for nombre, batch_size in variantes_gd.items():
    pesos, historial = gradiente_descendente(
        X_train, y_train, learning_rate=0.01, epocas=200,
        batch_size=batch_size, random_state=42,
    )
    resultados_gd[nombre] = (pesos, historial)
    print(f"{nombre}: MSE final (train) = {historial[-1]:.3f}")

# Gráfico Error vs Iteraciones (una curva por variante) que pide el enunciado
CARPETA_MODELO = BASE_DIR / "graficos_modelo"
os.makedirs(CARPETA_MODELO, exist_ok=True)

fig, ax = plt.subplots(figsize=(9, 5))
for nombre, (pesos, historial) in resultados_gd.items():
    ax.plot(historial, label=nombre)
ax.set_xlabel("Época")
ax.set_ylabel("MSE (train)")
ax.set_title("Gradiente descendente: Error vs Iteraciones")
ax.legend()
fig.savefig(CARPETA_MODELO / "gd_error_vs_iteraciones.png", dpi=100, bbox_inches="tight")
plt.close(fig)

# Comparamos los pesos del batch gradient descent contra los coeficientes de
# OLS: si convergió bien, deberían ser casi iguales (mismo problema, dos
# formas distintas de resolverlo: fórmula cerrada vs. iterando).
pesos_batch, _ = resultados_gd["Batch (todo el train)"]
comparacion_ols_gd = pd.DataFrame({
    "OLS": [modelo_ols.intercept_] + list(modelo_ols.coef_),
    "Gradiente (batch)": pesos_batch,
}, index=["intercepto"] + X_cols)
print("Comparación OLS vs gradiente descendente (batch):")
print(comparacion_ols_gd)

Batch (todo el train): MSE final (train) = 24.542
Estocástico (1 fila): MSE final (train) = 25.769
Mini-batch (32 filas): MSE final (train) = 23.179
Comparación OLS vs gradiente descendente (batch):
                  OLS  Gradiente (batch)
intercepto  22.210305          21.883521
CRIM        -1.028358          -0.804339
ZN           1.197945           0.706666
INDUS        0.121560          -0.335287
NOX         -1.967517          -0.750335
RM           2.132591           2.777526
AGE          0.113397          -0.162546
DIS         -3.138613          -2.061320
RAD          2.959250           1.053158
TAX         -2.000663          -0.435474
PTRATIO     -2.295013          -2.050451
B            0.823318           0.872114
LSTAT       -4.121450          -3.646885
CHAS         2.912953           2.080799


#### Conclusiones – Gradiente Descendente (batch)

**Interpretación de los coeficientes**
Como las variables están estandarizadas, los coeficientes se pueden comparar entre sí: cada uno indica cuánto cambia MEDV (en miles de USD) cuando la variable aumenta un desvío estándar y el resto se mantiene constante. El intercepto (21.88) corresponde aproximadamente al valor medio de MEDV.

- **LSTAT** es la variable más influyente (-3.65): a mayor proporción de población de bajo nivel socioeconómico, menor valor de la vivienda.
- **RM** (+2.78) es el principal factor positivo: más habitaciones implican viviendas más caras.
- **CHAS** (+2.08): estar junto al río Charles aumenta el valor de la vivienda.
- **DIS** (-2.06) y **PTRATIO** (-2.05) tienen un efecto negativo importante: la lejanía a los centros de empleo y más alumnos por docente reducen el precio.
- **RAD** (+1.05), **B** (+0.87) y **ZN** (+0.71) tienen un efecto positivo moderado.
- **CRIM** (-0.80) y **NOX** (-0.75) tienen un efecto negativo moderado: más criminalidad y más contaminación bajan el valor.
- **TAX** (-0.44), **INDUS** (-0.34) y **AGE** (-0.16) tienen coeficientes pequeños, así que su aporte al modelo es bajo.

**Conclusión:** el modelo entrenado con gradiente descendente identifica como variables más determinantes del precio el nivel socioeconómico (LSTAT), el tamaño de la vivienda (RM), la cercanía al río (CHAS), la distancia a los centros de empleo (DIS) y la calidad educativa (PTRATIO). Los signos de los coeficientes son coherentes con lo esperable. Las variables con coeficientes cercanos a cero (AGE, INDUS, TAX) podrían descartarse sin perder mucha capacidad predictiva, algo que se puede evaluar con regularización Lasso.



MODELO: RIDGE Y LASSO (regularización)

Ridge (L2) y Lasso (L1) agregan a la función de costo de OLS una
    penalización sobre el tamaño de los coeficientes, controlada por el
    hiperparámetro alpha (a mayor alpha, más penalización = coeficientes
    más chicos):

      - Ridge: penaliza la suma de los coeficientes al CUADRADO. Los achica
        pero nunca los lleva exactamente a 0.
      - Lasso: penaliza la suma de los coeficientes en VALOR ABSOLUTO.
        Puede llevar coeficientes exactamente a 0 -> hace selección de
        variables automática.

    Arrancamos con alpha=1.0 (default de sklearn) para comparar contra OLS;
    el tuneo de alpha es la consigna 5.

In [18]:
modelo_ridge = Ridge(alpha=1.0, random_state=42)
modelo_ridge.fit(X_train, y_train)
pred_train_ridge = modelo_ridge.predict(X_train)
pred_test_ridge = modelo_ridge.predict(X_test)

modelo_lasso = Lasso(alpha=1.0, random_state=42)
modelo_lasso.fit(X_train, y_train)
pred_train_lasso = modelo_lasso.predict(X_train)
pred_test_lasso = modelo_lasso.predict(X_test)


def reportar_metricas(nombre, y_tr, pred_tr, y_te, pred_te):
    """Imprime R², RMSE y MAE en train y test para un modelo ya entrenado."""
    rmse_tr = np.sqrt(mean_squared_error(y_tr, pred_tr))
    rmse_te = np.sqrt(mean_squared_error(y_te, pred_te))
    print(f"{nombre}:")
    print(f"  R²   train: {r2_score(y_tr, pred_tr):.3f}  |  R²   test: {r2_score(y_te, pred_te):.3f}")
    print(f"  RMSE train: {rmse_tr:.3f}  |  RMSE test: {rmse_te:.3f}")
    print(f"  MAE  train: {mean_absolute_error(y_tr, pred_tr):.3f}  |  MAE  test: {mean_absolute_error(y_te, pred_te):.3f}")


reportar_metricas("OLS", y_train, pred_train_ols, y_test, pred_test_ols)
reportar_metricas("Ridge (alpha=1.0)", y_train, pred_train_ridge, y_test, pred_test_ridge)
reportar_metricas("Lasso (alpha=1.0)", y_train, pred_train_lasso, y_test, pred_test_lasso)

# Comparamos los coeficientes de los 3 modelos lado a lado. Nos interesa
# sobre todo ver si Lasso llevó alguna variable a 0 (selección de
# variables) -> por EDA, las candidatas con menor relación con MEDV eran
# CHAS y B.
comparacion_coefs = pd.DataFrame({
    "OLS": modelo_ols.coef_,
    "Ridge": modelo_ridge.coef_,
    "Lasso": modelo_lasso.coef_,
}, index=X_cols).sort_values("OLS")
print("Comparación de coeficientes (OLS vs Ridge vs Lasso):")
print(comparacion_coefs)

variables_eliminadas = list(comparacion_coefs.index[comparacion_coefs["Lasso"] == 0])
print(f"Variables que Lasso llevó a 0: {variables_eliminadas}")

OLS:
  R²   train: 0.727  |  R²   test: 0.778
  RMSE train: 4.814  |  RMSE test: 4.284
  MAE  train: 3.346  |  MAE  test: 3.306
Ridge (alpha=1.0):
  R²   train: 0.727  |  R²   test: 0.778
  RMSE train: 4.814  |  RMSE test: 4.280
  MAE  train: 3.341  |  MAE  test: 3.297
Lasso (alpha=1.0):
  R²   train: 0.639  |  R²   test: 0.689
  RMSE train: 5.533  |  RMSE test: 5.069
  MAE  train: 3.834  |  MAE  test: 3.735
Comparación de coeficientes (OLS vs Ridge vs Lasso):
              OLS     Ridge     Lasso
LSTAT   -4.121450 -4.104590 -3.848355
DIS     -3.138613 -3.110543 -0.000000
PTRATIO -2.295013 -2.286689 -1.517180
TAX     -2.000663 -1.931868 -0.000000
NOX     -1.967517 -1.935998 -0.000000
CRIM    -1.028358 -1.020771 -0.000000
AGE      0.113397  0.105022 -0.000000
INDUS    0.121560  0.099455 -0.000000
B        0.823318  0.824086  0.021040
ZN       1.197945  1.181467  0.000000
RM       2.132591  2.145990  2.162102
CHAS     2.912953  2.822797  0.000000
RAD      2.959250  2.882250 -0.000000
Var

¿Cuál es mejor? Con alpha=1.0 (sin tunear), OLS y Ridge quedan prácticamente empatados y Lasso
queda bastante atrás:

      R² test   OLS=0.778   Ridge=0.778   Lasso=0.689
      RMSE test OLS=4.284   Ridge=4.280   Lasso=5.069
      MAE  test OLS=3.306   Ridge=3.297   Lasso=3.735

    Ridge (que solo achica coeficientes, sin eliminar ninguno) queda
    prácticamente igual que OLS: con alpha=1 la penalización es muy chica
    comparada con el tamaño del train, y la multicolinealidad que vimos en
    el VIF (TAX y RAD, moderada) no es tan fuerte como para que Ridge cambie
    mucho los coeficientes.

    Lasso, en cambio, llevó a 0 nada menos que 9 de las 13 variables
    (DIS, TAX, NOX, CRIM, AGE, INDUS, ZN, CHAS, RAD) y se quedó solo con
    LSTAT, PTRATIO, RM y B. Su R² bajó tanto en train (0.639 vs 0.727) como
    en test (0.689 vs 0.778) -> con alpha=1 Lasso está SUBajustando: tira
    variables que sí aportaban información.

    Conclusión: con estos hiperparámetros por defecto, OLS/Ridge son los
    mejores. alpha=1.0 es demasiado para Lasso. 
    Más adelante valuaremos probaremos modificando el alpha.

MODELO: ELASTIC NET (regularización L1 + L2)

Elastic Net combina las dos penalizaciones anteriores en una sola:

        penalización = l1_ratio * (L1, como Lasso) + (1 - l1_ratio) * (L2, como Ridge)

    l1_ratio=1.0 sería Lasso puro, l1_ratio=0.0 sería Ridge puro. Con
    l1_ratio=0.5 (default de sklearn) buscamos un punto intermedio: puede
    seguir llevando coeficientes a 0 (como Lasso) pero de forma más
    "suave" cuando hay variables correlacionadas entre sí -> en vez de
    quedarse arbitrariamente con una del grupo y tirar el resto a 0 (lo
    que hace Lasso), reparte el peso entre ellas. Relevante acá porque
    justamente eliminamos un cluster de variables correlacionadas
    (NOX-INDUS-AGE-DIS).

In [19]:
modelo_elastic = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42)
modelo_elastic.fit(X_train, y_train)
pred_train_elastic = modelo_elastic.predict(X_train)
pred_test_elastic = modelo_elastic.predict(X_test)

reportar_metricas("Elastic Net (alpha=1.0, l1_ratio=0.5)", y_train, pred_train_elastic, y_test, pred_test_elastic)

comparacion_coefs["ElasticNet"] = pd.Series(modelo_elastic.coef_, index=X_cols)   # por índice, no por posición (comparacion_coefs ya está reordenado)
print("Comparación de coeficientes (OLS vs Ridge vs Lasso vs Elastic Net):")
print(comparacion_coefs)

variables_eliminadas_en = list(comparacion_coefs.index[comparacion_coefs["ElasticNet"] == 0])
print(f"Variables que Elastic Net llevó a 0: {variables_eliminadas_en}")

Elastic Net (alpha=1.0, l1_ratio=0.5):
  R²   train: 0.616  |  R²   test: 0.673
  RMSE train: 5.705  |  RMSE test: 5.202
  MAE  train: 3.871  |  MAE  test: 3.778
Comparación de coeficientes (OLS vs Ridge vs Lasso vs Elastic Net):
              OLS     Ridge     Lasso  ElasticNet
LSTAT   -4.121450 -4.104590 -3.848355   -2.485041
DIS     -3.138613 -3.110543 -0.000000   -0.028845
PTRATIO -2.295013 -2.286689 -1.517180   -1.385080
TAX     -2.000663 -1.931868 -0.000000   -0.191817
NOX     -1.967517 -1.935998 -0.000000   -0.120351
CRIM    -1.028358 -1.020771 -0.000000   -0.384454
AGE      0.113397  0.105022 -0.000000   -0.000000
INDUS    0.121560  0.099455 -0.000000   -0.203586
B        0.823318  0.824086  0.021040    0.441063
ZN       1.197945  1.181467  0.000000    0.221220
RM       2.132591  2.145990  2.162102    2.120439
CHAS     2.912953  2.822797  0.000000    0.000000
RAD      2.959250  2.882250 -0.000000   -0.000000
Variables que Elastic Net llevó a 0: ['AGE', 'CHAS', 'RAD']


Con l1_ratio=0.5, Elastic Net se comportó como se esperaba: un punto intermedio entre Ridge y
Lasso en cuanto a la cantidad de variables que elimina.

      R² test    OLS=0.778  Ridge=0.778  Lasso=0.689  ElasticNet=0.673
      RMSE test  OLS=4.284  Ridge=4.280  Lasso=5.069  ElasticNet=5.202

    Solo eliminó 3 variables (AGE, CHAS y RAD) en vez de las 9 que eliminó
    Lasso, pero achicó mucho todos los coeficientes (por ejemplo LSTAT pasa
    de -4.1 en OLS a -2.5). Por eso su R² queda incluso un poco por debajo
    del de Lasso: con alpha=1.0 la penalización total es demasiado fuerte
    para este problema. Igual que con Lasso, hay que tunear alpha (y
    eventualmente l1_ratio) antes de sacar conclusiones.

¿CONSEGUIMOS UN BUEN FITTING?

In [20]:
resumen_modelos = pd.DataFrame({
    "OLS":        [r2_score(y_train, pred_train_ols),     r2_score(y_test, pred_test_ols)],
    "Ridge":      [r2_score(y_train, pred_train_ridge),   r2_score(y_test, pred_test_ridge)],
    "Lasso":      [r2_score(y_train, pred_train_lasso),   r2_score(y_test, pred_test_lasso)],
    "ElasticNet": [r2_score(y_train, pred_train_elastic), r2_score(y_test, pred_test_elastic)],
}, index=["R2_train", "R2_test"]).T
resumen_modelos["brecha_train_test"] = resumen_modelos["R2_train"] - resumen_modelos["R2_test"]
print(resumen_modelos)

            R2_train   R2_test  brecha_train_test
OLS         0.726595  0.777964          -0.051369
Ridge       0.726570  0.778396          -0.051826
Lasso       0.638893  0.689240          -0.050347
ElasticNet  0.616083  0.672664          -0.056581


Con los 4 modelos, el R² de test queda entre 0.67 y 0.78: el mejor (OLS/Ridge) explica cerca
del 78% de la variación de MEDV, con un error típico (RMSE) de unos 4.3 mil dólares. Es un
fitting RAZONABLE, aunque mejorable.

    ¿Por qué pedimos las métricas en train Y en test? Porque comparándolas
    se diagnostica el tipo de problema:
      - train alto y test mucho más bajo -> sobreajuste (el modelo memorizó train).
      - train y test bajos y parecidos -> subajuste (el modelo es demasiado simple).
    Acá train (0.727) y test (0.778) son parecidos -> NO hay sobreajuste.
    Que test sea incluso un poco MAYOR que train no significa que el modelo
    "generalice mejor que perfecto": es consecuencia de que este split en
    particular es algo más fácil que el promedio (ver el comentario de la
    división train/test). La validación cruzada sobre train, más abajo, da
    una estimación más estable (~0.69).

    ¿Qué le falta? Son todos modelos LINEALES, y ya vimos en el análisis
    descriptivo (scatter vs MEDV, matriz de Spearman vs Pearson) que varias
    de las relaciones más fuertes con MEDV son NO lineales: LSTAT, CRIM,
    DIS y RM muestran curvatura. Una recta tiene un techo para capturar
    esas relaciones -> por eso Lasso y Elastic Net, que quitan o achican
    variables, empeoran: el problema no es que el modelo tenga de más, sino
    que le falta flexibilidad. Lo retomamos al final, transformando esas
    variables.

OPTIMIZACION DE HIPERPARAMETROS

Gradiente descendente: variando el learning_rate

Usamos batch completo (para aislar el efecto del learning_rate del
    ruido de estocástico/mini-batch que ya vimos en la sección anterior) y
    probamos varios valores.

In [21]:
learning_rates = [0.0001, 0.001, 0.01, 0.1, 0.3, 0.5]
resultados_lr = {}
for lr in learning_rates:
    pesos, historial = gradiente_descendente(
        X_train, y_train, learning_rate=lr, epocas=300,
        batch_size=n_train, random_state=42,
    )
    resultados_lr[lr] = historial
    print(f"learning_rate={lr}: MSE final (train) = {historial[-1]:.3g}")

learning_rate=0.0001: MSE final (train) = 510
learning_rate=0.001: MSE final (train) = 184
learning_rate=0.01: MSE final (train) = 23.9
learning_rate=0.1: MSE final (train) = 23.2
learning_rate=0.3: MSE final (train) = 2.81e+256
learning_rate=0.5: MSE final (train) = inf


c:\Users\aldys\Desktop\AA\TP 1\TP_AA1_Sanchez_OdisioMartinelli_Zappala\.venv-1\Lib\site-packages\numpy\_core\_methods.py:132: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
C:\Users\aldys\AppData\Local\Temp\ipykernel_23428\2385333284.py:36: RuntimeWarning: overflow encountered in square
  historial_error.append(np.mean(error_epoca ** 2))


Con lr=0.0001 y lr=0.001 el error baja pero muy lento: a la época 300
    todavía está lejos del óptimo de OLS (~23). Con lr=0.01 y lr=0.1
    converge bien, cerca del óptimo. Con lr=0.3 y lr=0.5 EXPLOTA: el error
    crece sin control (llega a números astronómicos / infinito) en vez de
    bajar -> el paso es tan grande que en cada actualización se "pasa" del
    mínimo y cada vez rebota más lejos.

    Graficamos solo los learning_rate que no divergen: si incluyéramos
    0.3/0.5 en el mismo gráfico, su escala (números gigantes) aplastaría
    las demás curvas y no se vería nada.

In [22]:
fig, ax = plt.subplots(figsize=(9, 5))
for lr in [0.0001, 0.001, 0.01, 0.1]:
    ax.plot(resultados_lr[lr], label=f"lr={lr}")
ax.set_xlabel("Época")
ax.set_ylabel("MSE (train)")
ax.set_title("Gradiente descendente (batch): Error vs Iteraciones según learning_rate")
ax.legend()
fig.savefig(CARPETA_MODELO / "gd_learning_rate.png", dpi=100, bbox_inches="tight")
plt.close(fig)

print("learning_rate=0.3 y 0.5: el error DIVERGE (no se grafican, rompen la escala del resto).")

learning_rate=0.3 y 0.5: el error DIVERGE (no se grafican, rompen la escala del resto).


Gradiente descendente: variando la cantidad de épocas (con el mejor learning_rate)

Con lr=0.1 (el que mejor convergió arriba), vemos cuántas épocas hacen
    falta realmente para acercarse al óptimo de OLS, y si seguir
    entrenando después de ese punto sigue ayudando o ya no cambia nada
    (la curva se "aplana").

In [23]:
epocas_prueba = [10, 50, 100, 300]
for ep in epocas_prueba:
    _, historial = gradiente_descendente(
        X_train, y_train, learning_rate=0.1, epocas=ep,
        batch_size=n_train, random_state=42,
    )
    print(f"épocas={ep}: MSE final (train) = {historial[-1]:.3f}  (óptimo OLS ~ {rmse_train_ols**2:.3f})")

épocas=10: MSE final (train) = 31.242  (óptimo OLS ~ 23.177)
épocas=50: MSE final (train) = 23.454  (óptimo OLS ~ 23.177)
épocas=100: MSE final (train) = 23.226  (óptimo OLS ~ 23.177)
épocas=300: MSE final (train) = 23.177  (óptimo OLS ~ 23.177)


Ridge y Lasso: variando alpha

Barremos alpha en escala logarítmica (los valores de alpha razonables
    suelen abarcar varios órdenes de magnitud) y medimos, para cada uno:
      - R² por VALIDACIÓN CRUZADA (5-fold) SOBRE TRAIN -- no sobre test.
        Si eligiéramos alpha mirando el R² de test directamente, el test
        dejaría de ser una medida honesta de generalización (estaríamos
        "espiándolo" para elegir el modelo). Por eso partimos train en 5
        pliegues, entrenamos en 4 y validamos en el restante, rotando, y
        promediamos -> así elegimos alpha sin tocar test para nada. Recién
        en la sección de "comparación final" evaluamos UNA VEZ en test,
        con el alpha ya elegido.
        La imputación y la estandarización se hacen DENTRO de cada fold
        (con un Pipeline): si las ajustáramos antes con todo train, el fold
        de validación participaría en calcular las medias, desvíos y
        vecinos del KNN, y dejaría de ser un dato "no visto".
      - cantidad de coeficientes que Lasso llevó a 0 (su "poda" de
        variables, para responder ¿en qué punto empieza a eliminar?)

In [24]:
alphas = np.logspace(-3, 3, 13)   # 0.001, ..., 1000 (13 valores log-espaciados)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

Incluimos también a Elastic Net (l1_ratio=0.5 fijo, variando solo
    alpha) en el MISMO barrido con CV -- si tuneamos Ridge y Lasso pero
    dejamos Elastic Net en su alpha por defecto, la comparación final no
    sería justa (estaríamos comparando dos modelos optimizados contra uno
    que no).

In [25]:
# Preprocesamiento "empaquetado": el mismo que hicimos antes (moda para CHAS,
# estandarización + KNNImputer para el resto), pero ahora cross_val_score lo
# vuelve a ajustar (fit) DENTRO de cada fold, usando solo los 4 folds de
# entrenamiento. Así el fold de validación no participa en calcular medias,
# desvíos, modas ni vecinos -> es tan "desconocido" como el test.
preprocesamiento = ColumnTransformer([
    ("num", make_pipeline(StandardScaler(), KNNImputer(n_neighbors=5)), list(columnas_knn)),
    ("chas", SimpleImputer(strategy="most_frequent"), ["CHAS"]),
])

# Train SIN imputar ni escalar (con sus NaN): el pipeline se encarga de eso
X_train_crudo = df_train_crudo[list(columnas_knn) + ["CHAS"]].astype(float)
y_train_crudo = df_train_crudo["MEDV"].astype(float)

r2_cv_ridge, r2_cv_lasso, r2_cv_elastic = [], [], []
n_variables_lasso = []

for alpha in alphas:
    pipe_ridge = make_pipeline(preprocesamiento, Ridge(alpha=alpha, random_state=42))
    pipe_lasso = make_pipeline(preprocesamiento, Lasso(alpha=alpha, random_state=42, max_iter=10000))
    pipe_elastic = make_pipeline(preprocesamiento, ElasticNet(alpha=alpha, l1_ratio=0.5, random_state=42, max_iter=10000))

    r2_cv_ridge.append(
        cross_val_score(pipe_ridge, X_train_crudo, y_train_crudo, cv=kfold, scoring="r2").mean()
    )
    r2_cv_lasso.append(
        cross_val_score(pipe_lasso, X_train_crudo, y_train_crudo, cv=kfold, scoring="r2").mean()
    )
    r2_cv_elastic.append(
        cross_val_score(pipe_elastic, X_train_crudo, y_train_crudo, cv=kfold, scoring="r2").mean()
    )
    # La cantidad de variables activas la miramos reentrenando con TODO
    # train (no por fold) -- es solo para describir el modelo, no para
    # elegir alpha, así que no hace falta cross-validarla.
    m_lasso_full = Lasso(alpha=alpha, random_state=42, max_iter=10000).fit(X_train, y_train)
    n_variables_lasso.append(np.sum(m_lasso_full.coef_ != 0))

tabla_alpha = pd.DataFrame({
    "alpha": alphas,
    "R2_cv_Ridge": r2_cv_ridge,
    "R2_cv_Lasso": r2_cv_lasso,
    "R2_cv_ElasticNet": r2_cv_elastic,
    "variables_activas_Lasso": n_variables_lasso,
})
print(tabla_alpha.to_string(index=False))

      alpha  R2_cv_Ridge  R2_cv_Lasso  R2_cv_ElasticNet  variables_activas_Lasso
   0.001000     0.688608     0.688682          0.688781                       13
   0.003162     0.688610     0.688838          0.689133                       13
   0.010000     0.688616     0.689286          0.690068                       13
   0.031623     0.688635     0.690236          0.691640                       11
   0.100000     0.688693     0.688007          0.689690                       11
   0.316228     0.688872     0.660401          0.666675                        8
   1.000000     0.689393     0.616466          0.602792                        4
   3.162278     0.690654     0.446565          0.434596                        3
  10.000000     0.692256    -0.009696          0.055724                        0
  31.622777     0.689228    -0.009696         -0.009696                        0
 100.000000     0.668090    -0.009696         -0.009696                        0
 316.227766     0.599917    

In [26]:
# Gráfico: R² de validación cruzada vs alpha (escala log en X)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(alphas, r2_cv_ridge, marker="o", label="Ridge")
ax.plot(alphas, r2_cv_lasso, marker="o", label="Lasso")
ax.plot(alphas, r2_cv_elastic, marker="o", label="Elastic Net (l1_ratio=0.5)")
ax.set_xscale("log")
ax.set_xlabel("alpha (escala log)")
ax.set_ylabel("R² (validación cruzada, 5-fold sobre train)")
ax.set_title("R² de validación cruzada según alpha — Ridge vs Lasso vs Elastic Net")
ax.legend()
fig.savefig(CARPETA_MODELO / "ridge_lasso_alpha.png", dpi=100, bbox_inches="tight")
plt.close(fig)

# Gráfico: cantidad de variables activas de Lasso vs alpha (su "camino de
# selección" — a qué alpha empieza a eliminar variables)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(alphas, n_variables_lasso, marker="o", color="tab:orange")
ax.set_xscale("log")
ax.set_xlabel("alpha (escala log)")
ax.set_ylabel("Cantidad de variables con coeficiente ≠ 0")
ax.set_title("Lasso: cuántas variables sobreviven según alpha")
fig.savefig(CARPETA_MODELO / "lasso_seleccion_variables.png", dpi=100, bbox_inches="tight")
plt.close(fig)

¿Qué observamos?

    Con R² de VALIDACIÓN CRUZADA (no de test) como criterio, el óptimo de
    Lasso está en alpha=0.0316 (R²_cv=0.690) -- muy poca regularización,
    con 11 de las 13 variables activas. Lasso empieza a "podar" variables
    alrededor de alpha~0.03 (de 13 a 11), la poda fuerte es entre
    alpha=0.1 y alpha=1 (de 11 a 4), y desde alpha=10 en adelante ya
    eliminó todo (0 variables, solo predice la media -> R²_cv negativo).
    Es decir: según CV, NO conviene que Lasso pode tanto -- el alpha=1.0
    que usamos al principio (que eliminaba 9 variables) está en una zona
    donde el R²_cv ya viene cayendo (0.616), no en el óptimo.

    Ridge, en cambio, tiene su óptimo en alpha=10 (R²_cv=0.692) -- bastante
    más regularización que el alpha=1.0 por defecto (R²_cv=0.689 ahí). Ridge
    tolera alphas mucho más grandes que Lasso porque nunca elimina
    variables del todo: recién a partir de alpha~100 empieza a caer fuerte.

    Elastic Net (l1_ratio=0.5) tiene su óptimo también en alpha=0.0316
    (R²_cv=0.692) -- con tan poca regularización, se comporta casi como OLS.

    Conclusión importante: el R²_cv apenas varía entre alpha=0.001 y
    alpha=0.1 para los tres modelos (todos rondando 0.69) -- hay una meseta
    ancha, no un pico marcado. Para este dataset la regularización ayuda muy
    poco: el modelo no está sobreajustando, así que no hay varianza que
    reducir. El techo lo pone la forma LINEAL del modelo, no la cantidad de
    variables.

COMPARACION FINAL DE MODELOS

Recopilamos TODOS los modelos, ya con los mejores hiperparámetros
    encontrados en esta sección, y los comparamos con una misma métrica.
    Elegimos RMSE de test como métrica principal para decidir el "mejor":
    a diferencia de R² (que es relativo, 0 a 1), RMSE queda en las mismas
    unidades que MEDV (miles de dólares) -> es directamente interpretable
    ("nos equivocamos en promedio X mil dólares"), y penaliza más los
    errores grandes que MAE (relevante si nos importa evitar
    predicciones muy desacertadas en casas caras/baratas).

In [27]:
# Reentrenamos Ridge, Lasso y Elastic Net con su mejor alpha según
# VALIDACIÓN CRUZADA sobre train (no según test -- test recién se usa acá
# abajo, una sola vez, para reportar el resultado final de cada modelo ya
# elegido).
mejor_alpha_ridge = tabla_alpha.loc[tabla_alpha["R2_cv_Ridge"].idxmax(), "alpha"]
mejor_alpha_lasso = tabla_alpha.loc[tabla_alpha["R2_cv_Lasso"].idxmax(), "alpha"]
mejor_alpha_elastic = tabla_alpha.loc[tabla_alpha["R2_cv_ElasticNet"].idxmax(), "alpha"]

modelo_ridge_tuneado = Ridge(alpha=mejor_alpha_ridge, random_state=42).fit(X_train, y_train)
modelo_lasso_tuneado = Lasso(alpha=mejor_alpha_lasso, random_state=42, max_iter=10000).fit(X_train, y_train)
modelo_elastic_tuneado = ElasticNet(alpha=mejor_alpha_elastic, l1_ratio=0.5, random_state=42, max_iter=10000).fit(X_train, y_train)

# Gradiente descendente con el mejor learning_rate/épocas encontrados (batch)
pesos_gd_tuneado, _ = gradiente_descendente(
    X_train, y_train, learning_rate=0.1, epocas=300, batch_size=n_train, random_state=42,
)
X_train_b = np.hstack([np.ones((X_train.shape[0], 1)), X_train])
X_test_b = np.hstack([np.ones((X_test.shape[0], 1)), X_test])
pred_train_gd = X_train_b @ pesos_gd_tuneado
pred_test_gd = X_test_b @ pesos_gd_tuneado

modelos_finales = {
    "OLS": (pred_train_ols, pred_test_ols),
    "Gradiente descendente (lr=0.1, 300 épocas)": (pred_train_gd, pred_test_gd),
    f"Ridge (alpha={mejor_alpha_ridge:.3g})": (modelo_ridge_tuneado.predict(X_train), modelo_ridge_tuneado.predict(X_test)),
    f"Lasso (alpha={mejor_alpha_lasso:.3g})": (modelo_lasso_tuneado.predict(X_train), modelo_lasso_tuneado.predict(X_test)),
    f"Elastic Net (alpha={mejor_alpha_elastic:.3g}, l1_ratio=0.5)": (modelo_elastic_tuneado.predict(X_train), modelo_elastic_tuneado.predict(X_test)),
}

comparacion_final = pd.DataFrame({
    nombre: {
        "R2_train": r2_score(y_train, pred_tr),
        "R2_test": r2_score(y_test, pred_te),
        "RMSE_train": np.sqrt(mean_squared_error(y_train, pred_tr)),
        "RMSE_test": np.sqrt(mean_squared_error(y_test, pred_te)),
        "MAE_test": mean_absolute_error(y_test, pred_te),
    }
    for nombre, (pred_tr, pred_te) in modelos_finales.items()
}).T.sort_values("RMSE_test")

print(comparacion_final)
print(f"\nMejor modelo según RMSE de test: {comparacion_final.index[0]}")

                                            R2_train   R2_test  RMSE_train  \
Ridge (alpha=10)                            0.724918  0.780274    4.828967   
Elastic Net (alpha=0.0316, l1_ratio=0.5)    0.725098  0.780037    4.827385   
Lasso (alpha=0.0316)                        0.725897  0.779144    4.820361   
Gradiente descendente (lr=0.1, 300 épocas)  0.726593  0.778034    4.814242   
OLS                                         0.726595  0.777964    4.814224   

                                            RMSE_test  MAE_test  
Ridge (alpha=10)                             4.262022  3.233708  
Elastic Net (alpha=0.0316, l1_ratio=0.5)     4.264326  3.240330  
Lasso (alpha=0.0316)                         4.272967  3.269729  
Gradiente descendente (lr=0.1, 300 épocas)   4.283697  3.305195  
OLS                                          4.284373  3.305713  

Mejor modelo según RMSE de test: Ridge (alpha=10)


Resultado final, con Ridge/Lasso/ElasticNet tuneados por VALIDACIÓN
    CRUZADA sobre train (RMSE test, de menor a mayor):

      Ridge (alpha=10)                         R2_test=0.780  RMSE_test=4.262  MAE_test=3.234
      Elastic Net (alpha=0.0316, l1_ratio=.5)  R2_test=0.780  RMSE_test=4.264  MAE_test=3.240
      Lasso (alpha=0.0316)                     R2_test=0.779  RMSE_test=4.273  MAE_test=3.270
      Gradiente descendente (lr=0.1)           R2_test=0.778  RMSE_test=4.284  MAE_test=3.305
      OLS                                      R2_test=0.778  RMSE_test=4.284  MAE_test=3.306

    GANADOR: Ridge con alpha=10, pero por muy poco. Las diferencias entre
    los 5 modelos son de centésimas de RMSE (4.26 vs 4.28): en la práctica
    están empatados, y con otro split de train/test el orden podría
    cambiar. Esto es coherente con lo que vimos en CV: una vez tuneados,
    los tres regularizados eligen alphas que los dejan muy cerca de OLS,
    porque el modelo no estaba sobreajustando.

    El gradiente descendente llega prácticamente al mismo resultado que
    OLS (mismo problema resuelto de dos formas: iterando vs. fórmula
    cerrada), lo que confirma que convergió.

MEJORA: TRANSFORMACIONES NO LINEALES DE LAS VARIABLES

En "¿Conseguimos un buen fitting?" vimos que el límite principal de los modelos es que varias
relaciones con MEDV son curvas (en los scatter vs MEDV y en la diferencia entre Spearman y
Pearson). Una regresión lineal puede capturar esas curvas si le damos las variables
transformadas: el modelo sigue siendo LINEAL en sus coeficientes, solo cambian las columnas
de entrada.

Elegimos las transformaciones mirando el análisis descriptivo (nunca el test):
  - log(LSTAT): MEDV cae muy rápido con LSTAT al principio y después se aplana -> forma logarítmica.
  - log(CRIM) y log(DIS): muy asimétricas, con cola larga a la derecha; el log las "comprime".
  - RM²: MEDV crece más que linealmente con la cantidad de habitaciones (casas grandes valen
    mucho más).

Para decidir si la mejora es real, comparamos por validación cruzada sobre train (con el mismo
Pipeline, para que imputación y escalado se ajusten dentro de cada fold) las variables
originales contra las transformadas, usando Ridge en todo el rango de alphas.

In [28]:
def agregar_transformaciones(X):
    """Reemplaza LSTAT, CRIM y DIS por su logaritmo y agrega RM². Los NaN siguen siendo NaN
    (se imputan después, dentro del pipeline)."""
    X = X.copy()
    X["LSTAT"] = np.log(X["LSTAT"])
    X["CRIM"] = np.log(X["CRIM"])
    X["DIS"] = np.log(X["DIS"])
    X["RM2"] = X["RM"] ** 2
    return X

columnas_transf = list(columnas_knn) + ["RM2"]

preprocesamiento_transf = ColumnTransformer([
    ("num", make_pipeline(StandardScaler(), KNNImputer(n_neighbors=5)), columnas_transf),
    ("chas", SimpleImputer(strategy="most_frequent"), ["CHAS"]),
])


def pipeline_transf(modelo):
    """Transformaciones -> imputación + escalado -> modelo, todo ajustado solo con los datos de fit."""
    return make_pipeline(FunctionTransformer(agregar_transformaciones), preprocesamiento_transf, modelo)


r2_cv_ridge_transf = [
    cross_val_score(pipeline_transf(Ridge(alpha=alpha, random_state=42)),
                    X_train_crudo, y_train_crudo, cv=kfold, scoring="r2").mean()
    for alpha in alphas
]

tabla_transf = pd.DataFrame({
    "alpha": alphas,
    "R2_cv_Ridge_originales": r2_cv_ridge,
    "R2_cv_Ridge_transformadas": r2_cv_ridge_transf,
})
print(tabla_transf.to_string(index=False))

mejor_alpha_transf = tabla_transf.loc[tabla_transf["R2_cv_Ridge_transformadas"].idxmax(), "alpha"]
print(f"\nMejor alpha (variables transformadas, según CV): {mejor_alpha_transf:.3g}")

      alpha  R2_cv_Ridge_originales  R2_cv_Ridge_transformadas
   0.001000                0.688608                   0.762847
   0.003162                0.688610                   0.762852
   0.010000                0.688616                   0.762869
   0.031623                0.688635                   0.762914
   0.100000                0.688693                   0.762995
   0.316228                0.688872                   0.762833
   1.000000                0.689393                   0.760887
   3.162278                0.690654                   0.755934
  10.000000                0.692256                   0.751475
  31.622777                0.689228                   0.744241
 100.000000                0.668090                   0.716116
 316.227766                0.599917                   0.649199
1000.000000                0.461216                   0.521772

Mejor alpha (variables transformadas, según CV): 0.1


In [29]:
# Entrenamos el modelo elegido con TODO train y lo evaluamos UNA vez en test.
# Al test le pasamos sus datos crudos: el pipeline le aplica las mismas
# transformaciones, la imputación y el escalado aprendidos de train.
X_test_crudo = df_test_crudo[list(columnas_knn) + ["CHAS"]].astype(float)
y_test_crudo = df_test_crudo["MEDV"].astype(float)

modelo_transf = pipeline_transf(Ridge(alpha=mejor_alpha_transf, random_state=42))
modelo_transf.fit(X_train_crudo, y_train_crudo)

pred_train_transf = modelo_transf.predict(X_train_crudo)
pred_test_transf = modelo_transf.predict(X_test_crudo)
nombre_transf = f"Ridge + transformaciones (alpha={mejor_alpha_transf:.3g})"
reportar_metricas(nombre_transf, y_train_crudo, pred_train_transf, y_test_crudo, pred_test_transf)

# Lo sumamos a la tabla de comparación final
comparacion_final.loc[nombre_transf] = {
    "R2_train": r2_score(y_train_crudo, pred_train_transf),
    "R2_test": r2_score(y_test_crudo, pred_test_transf),
    "RMSE_train": np.sqrt(mean_squared_error(y_train_crudo, pred_train_transf)),
    "RMSE_test": np.sqrt(mean_squared_error(y_test_crudo, pred_test_transf)),
    "MAE_test": mean_absolute_error(y_test_crudo, pred_test_transf),
}
comparacion_final = comparacion_final.sort_values("RMSE_test")
print()
print(comparacion_final)
print(f"\nMejor modelo según RMSE de test: {comparacion_final.index[0]}")

Ridge + transformaciones (alpha=0.1):
  R²   train: 0.803  |  R²   test: 0.866
  RMSE train: 4.090  |  RMSE test: 3.327
  MAE  train: 2.883  |  MAE  test: 2.424

                                            R2_train   R2_test  RMSE_train  \
Ridge + transformaciones (alpha=0.1)        0.802646  0.866086    4.090211   
Ridge (alpha=10)                            0.724918  0.780274    4.828967   
Elastic Net (alpha=0.0316, l1_ratio=0.5)    0.725098  0.780037    4.827385   
Lasso (alpha=0.0316)                        0.725897  0.779144    4.820361   
Gradiente descendente (lr=0.1, 300 épocas)  0.726593  0.778034    4.814242   
OLS                                         0.726595  0.777964    4.814224   

                                            RMSE_test  MAE_test  
Ridge + transformaciones (alpha=0.1)         3.327276  2.424355  
Ridge (alpha=10)                             4.262022  3.233708  
Elastic Net (alpha=0.0316, l1_ratio=0.5)     4.264326  3.240330  
Lasso (alpha=0.0316)       

¿Qué observamos?

    Con las variables transformadas, el R² de validación cruzada sube de
    ~0.69 a ~0.76 en todo el rango de alphas razonables -> la mejora es
    real y no depende de un alpha puntual. El mejor alpha según CV es 0.1
    (muy poca regularización, igual que antes).

    En test, este modelo pasa a R²=0.866 y RMSE=3.33 (vs 0.780 y 4.26 del
    mejor modelo con las variables originales): baja el error típico en
    casi mil dólares. Ahora train (0.803) también sube: el modelo
    anterior estaba SUBajustando, y darle las formas curvas de LSTAT, CRIM,
    DIS y RM le permitió explicar más de la variación de MEDV.

    Como en todo el trabajo, el R² de test de este split es algo optimista
    (split más fácil que el promedio); la estimación más estable es la de
    CV, ~0.76.

    Es importante que el modelo sigue siendo una regresión lineal: lo que
    cambió fueron las columnas de entrada, no el método. Y las
    transformaciones se eligieron con el análisis descriptivo y se
    validaron con CV sobre train; el test se usó una sola vez, al final.

    MEJOR MODELO FINAL: Ridge (alpha=0.1) con variables transformadas.

CONCLUSION DEL TRABAJO

1) LIMPIEZA Y PREPARACIÓN DE DATOS
    Partimos de 556 filas con nulos dispersos (hasta 28 por columna) y
    algunos valores extremos. Los valores extremos reales (por ejemplo
    barrios con criminalidad muy alta) los CONSERVAMOS. En cambio,
    encontramos 48 filas con valores IMPOSIBLES según la definición de las
    variables (RAD no entero, TAX con decimales, MEDV con 6 decimales...),
    casi siempre en varias columnas a la vez: las descartamos con una regla
    fija, y quedaron las 506 filas del dataset original de Boston. Este fue
    el cambio que más impacto tuvo en todo el trabajo: con esas filas, los
    modelos lineales no pasaban de R²≈0.4 en test.

    CHAS se trató como categórica (ya viene codificada 0/1). Para los
    valores faltantes, separamos primero train/test (para no filtrar
    información de test hacia train), imputamos CHAS por la moda y el resto
    con KNNImputer sobre variables estandarizadas; imputers y scaler se
    ajustan solo con train. El VIF mostró multicolinealidad moderada entre
    TAX y RAD (VIF 8.4 y 7.1, < 10), y decidimos conservar ambas.

2) MODELOS
    Implementamos y comparamos regresión lineal (OLS), gradiente
    descendente (batch/estocástico/mini-batch) y regularización Ridge,
    Lasso y Elastic Net. Para elegir el alpha de cada uno usamos validación
    cruzada (5-fold) SOLO sobre train, con la imputación y el escalado
    dentro de cada fold (Pipeline) para que el fold de validación no
    participe en el preprocesamiento. Una vez tuneados, los cinco modelos
    quedaron prácticamente empatados (R² test ≈ 0.78, RMSE ≈ 4.27): el
    modelo no sobreajustaba, así que la regularización no tenía mucho que
    corregir. Con alpha=1 (sin tunear), Lasso y Elastic Net incluso
    empeoraban por eliminar variables útiles.

3) MEJORA Y MEJOR MODELO
    El límite de los modelos anteriores era la forma lineal: varias
    relaciones con MEDV son curvas. Transformando LSTAT, CRIM y DIS con
    logaritmo y agregando RM², el R² de CV subió de ~0.69 a ~0.76, y en
    test el mejor modelo (Ridge, alpha=0.1, variables transformadas) llegó
    a R²=0.866 y RMSE=3.33 mil dólares. Es un buen fitting: train y test
    son parecidos (no hay sobreajuste) y el modelo explica más de tres
    cuartas partes de la variación del precio.